In [2]:
import math
import torch 
import pandas as pd 
import torch 
from torch import nn 
from d2l import torch as d2l

In [3]:
class PositionWiseFNN(nn.Module):  # @save
    """
    A 2 layer Feedforward network, applied on all sequence positions.

    Input shape: (batch_size, num time steps or sequnce length, number of hidden units)
    Output shape: (batch_size, num_time steps, ffn_num_outputs)
    """

    def __init__(self, ffn_num_hiddens, ffn_num_outputs):
        super().__init__()
        self.dense1 = nn.LazyLinear(ffn_num_hiddens)
        self.relu = nn.ReLU()
        self.dense2 = nn.LazyLinear(ffn_num_outputs)

    def forward(self, X):
        return self.dense2(self.relu(self.dense1(X)))

In [4]:
ffn = PositionWiseFNN(4, 8)
ffn.eval()
ffn(torch.ones((2, 3, 4)))[0]

/home/sih/miniconda3/envs/d2l/lib/python3.9/site-packages/torch/nn/modules/lazy.py:180: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '


tensor([[ 0.6627,  0.7273,  0.3921, -0.5729,  0.3611,  0.9045, -0.4904,  0.2219],
        [ 0.6627,  0.7273,  0.3921, -0.5729,  0.3611,  0.9045, -0.4904,  0.2219],
        [ 0.6627,  0.7273,  0.3921, -0.5729,  0.3611,  0.9045, -0.4904,  0.2219]],
       grad_fn=<SelectBackward0>)

In [5]:
class AddNorm(nn.Module):  # @save
    """Residual connection followed by layer normalizations"""

    def __init__(self, norm_shape, dropout):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        # Layer Normalization
        self.ln = nn.LayerNorm(norm_shape)

    def forward(self, X, Y):
        return self.ln(self.dropout(Y) + X)

# Encoder Block 

In [6]:
class TransformerEncoderBlock(nn.Module):  # @save
    """
    The tranformer encoder block
    Doesnot change the shape of the input X
    """

    def __init__(
        self, num_hiddens, ffn_num_hiddens, num_heads, dropout, use_bias=False
    ):
        super().__init__()
        self.attention = d2l.MultiHeadAttention(
            num_hiddens, num_heads, dropout, use_bias
        )
        self.addnorm1 = AddNorm(num_hiddens, dropout)
        self.ffn = PositionWiseFNN(ffn_num_hiddens, num_hiddens)
        self.addnorm2 = AddNorm(num_hiddens, dropout)

    def forward(self, X, valid_lens):
        # Notice that X is both the query, key and value at the same time
        Y = self.addnorm1(X, self.attention(X, X, X, valid_lens))
        return self.addnorm2(Y, self.ffn(Y))

In [7]:
X = torch.ones((2, 100, 24))
valid_lens = torch.tensor([3, 2])
encoder_blk = TransformerEncoderBlock(24, 48, 8, 0.5)
encoder_blk.eval()
# No change in shape
d2l.check_shape(encoder_blk(X, valid_lens), X.shape)

In [8]:
class TransformerEncoder(d2l.Encoder):  # @save
    """
    Formed by stacking the encoder block n-times
    """

    def __init__(
        self,
        vocab_size,
        num_hiddens,
        ffn_num_hiddens,
        num_heads,
        num_blocks,
        dropout,
        use_bias=False,
    ):
        super().__init__()
        self.num_hiddens = num_hiddens
        self.embedding = nn.Embedding(vocab_size, num_hiddens)
        self.pos_encoding = d2l.PositionalEncoding(num_hiddens, dropout)
        # The different encoder blocks are sequentially arranged
        self.blocks = nn.Sequential()
        for i in range(num_blocks):
            self.blocks.add_module(
                "block" + str(i),
                TransformerEncoderBlock(
                    num_hiddens, ffn_num_hiddens, num_heads, dropout, use_bias
                ),
            )
    def forward(self, X, valid_lens):
        # Since the positional encoding values are btn -1 and 1, the embedding values are multiplied
        # by the square root of the embedding dimenstion to rescale before summing ip
        # the input embedding and positional encoding
        X = self.pos_encoding(self.embedding(X) * math.sqrt(self.num_hiddens))
        
